In [2]:
import os
import shutil
import numpy as np
import subprocess # <- MOLTO meglio di os.system!

# Inserisci qui il nome ESATTO del tuo eseguibile C++
# Su Windows solitamente serve ".\" davanti se è nella stessa cartella, o magari si chiama "main.exe"
ESEGUIBILE = ".\\simulator.exe" 

# Array delle temperature da 2.0 a 0.5 scendendo a passi di 0.1
temperatures = np.round(np.arange(2.0, 0.4, -0.1), 1)

def scrivi_properties(h_field):
    """Scrive properties.dat a seconda se stiamo misurando a h=0 o h>0"""
    with open("../INPUT/properties.dat", "w") as f:
        if h_field == 0.0:
            f.write("TOTAL_ENERGY\n")
            f.write("SPECIFIC_HEAT\n")
            f.write("SUSCEPTIBILITY\n")
        else:
            f.write("MAGNETIZATION\n")
        f.write("ENDPROPERTIES\n")

def scrivi_input(sim_type, h_field, restart, temp, nblocks, nsteps):
    """Scrive input.dat rispettando ESATTAMENTE la formattazione richiesta"""
    with open("../INPUT/input.dat", "w") as f:
        f.write(f"SIMULATION_TYPE        {sim_type}    1.0    {h_field}\n")
        f.write(f"RESTART                {restart}\n")
        f.write(f"TEMP                   {temp:.1f}\n")
        f.write(f"NPART                  50\n")
        f.write(f"RHO                    1.0\n")
        f.write(f"R_CUT                  0.0\n")
        f.write(f"DELTA                  0.0\n")
        f.write(f"NBLOCKS                {nblocks}\n")
        f.write(f"NSTEPS                 {nsteps}\n")
        f.write("\nENDINPUT\n")

def pulisci_output():
    """Elimina i file .dat vecchi o della termalizzazione per evitare append mischiati"""
    files = ["total_energy.dat", "specific_heat.dat", "susceptibility.dat", "magnetization.dat", "acceptance.dat"]
    for file in files:
        path = f"../OUTPUT/{file}"
        if os.path.exists(path):
            os.remove(path)

def lancia_cpp():
    """Lancia l'eseguibile e cattura eventuali crash"""
    try:
        # check=True fa sì che se il C++ crascia, Python si ferma subito e ti dice perché
        subprocess.run([ESEGUIBILE], check=True)
    except FileNotFoundError:
        print(f"\n❌ ERRORE: Non trovo l'eseguibile '{ESEGUIBILE}'!")
        print("Verifica che il nome sia corretto e di trovarti nella cartella giusta.")
        exit(1)
    except subprocess.CalledProcessError as e:
        print(f"\n❌ ERRORE C++: La simulazione è crasciata (Codice di uscita: {e.returncode})")
        exit(1)

def esegui_simulazione(sim_type, h_field, prefisso_nome):
    scrivi_properties(h_field)
    
    restart_iniziale = 0 
    
    for T in temperatures:
        print(f"\n=== {prefisso_nome} | T = {T:.1f} ===")
        
        # ---------------------------------------------------
        # FASE 1: TERMALIZZAZIONE
        # ---------------------------------------------------
        print("  -> Termalizzazione in corso...")
        pulisci_output()

        # Correzione: 1 blocco da 50000 step, NON 1 milione di blocchi da 1!
        # Così il C++ fa i calcoli in RAM ed è istantaneo, senza ingolfare l'hard disk
        scrivi_input(sim_type, h_field, restart_iniziale, T, nblocks=1, nsteps=50000)
        
        lancia_cpp()
        
        # Copia la config
        shutil.copy("../OUTPUT/CONFIG/config.spin", "../INPUT/CONFIG/config.spin")
        
        # ---------------------------------------------------
        # FASE 2: MISURA
        # ---------------------------------------------------
        print("  -> Misura in corso...")
        pulisci_output() 
        
        scrivi_input(sim_type, h_field, 1, T, nblocks=100, nsteps=10000) # 100 blocchi x 10000 step
        
        lancia_cpp()
        
        # ---------------------------------------------------
        # FASE 3: SALVATAGGIO DEI RISULTATI
        # ---------------------------------------------------
        # Un ulteriore controllo di sicurezza: rinomina SOLO se il file esiste
        try:
            if h_field == 0.0:
                os.rename("../OUTPUT/total_energy.dat", f"../OUTPUT/total_energy_{prefisso_nome}_{T:.1f}.dat")
                os.rename("../OUTPUT/specific_heat.dat", f"../OUTPUT/specific_heat_{prefisso_nome}_{T:.1f}.dat")
                os.rename("../OUTPUT/susceptibility.dat", f"../OUTPUT/susceptibility_{prefisso_nome}_{T:.1f}.dat")
            else:
                os.rename("../OUTPUT/magnetization.dat", f"../OUTPUT/magnetization_{prefisso_nome}_{T:.1f}.dat")
        except FileNotFoundError as e:
            print(f"❌ ERRORE CRITICO: Il file non è stato generato dal C++! Dettagli: {e}")
            exit(1)
            
        # Preparo il sistema per la prossima temperatura
        shutil.copy("../OUTPUT/CONFIG/config.spin", "../INPUT/CONFIG/config.spin")
        restart_iniziale = 1

# ==============================================================
# ESECUZIONE DEL SCRIPT
# ==============================================================

# 1. Metropolis con h=0
esegui_simulazione(sim_type=2, h_field=0.0, prefisso_nome="Metro_h0")

# 2. Metropolis con h=0.02
esegui_simulazione(sim_type=2, h_field=0.02, prefisso_nome="Metro_h02")

# 3. Gibbs con h=0
esegui_simulazione(sim_type=3, h_field=0.0, prefisso_nome="Gibbs_h0")

# 4. Gibbs con h=0.02
esegui_simulazione(sim_type=3, h_field=0.02, prefisso_nome="Gibbs_h02")

print("\n🚀 Tutte le simulazioni completate con successo!")


=== Metro_h0 | T = 2.0 ===
  -> Termalizzazione in corso...

❌ ERRORE C++: La simulazione è crasciata (Codice di uscita: 1)
  -> Misura in corso...

❌ ERRORE C++: La simulazione è crasciata (Codice di uscita: 1)
❌ ERRORE CRITICO: Il file non è stato generato dal C++! Dettagli: [WinError 2] Impossibile trovare il file specificato: '../OUTPUT/total_energy.dat' -> '../OUTPUT/total_energy_Metro_h0_2.0.dat'

=== Metro_h0 | T = 1.9 ===
  -> Termalizzazione in corso...

❌ ERRORE C++: La simulazione è crasciata (Codice di uscita: 1)
  -> Misura in corso...

❌ ERRORE C++: La simulazione è crasciata (Codice di uscita: 1)
❌ ERRORE CRITICO: Il file non è stato generato dal C++! Dettagli: [WinError 2] Impossibile trovare il file specificato: '../OUTPUT/total_energy.dat' -> '../OUTPUT/total_energy_Metro_h0_1.9.dat'

=== Metro_h0 | T = 1.8 ===
  -> Termalizzazione in corso...

❌ ERRORE C++: La simulazione è crasciata (Codice di uscita: 1)
  -> Misura in corso...

❌ ERRORE C++: La simulazione è crasc